In [3]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))
print(ROOT_DIR)

c:\Users\kuchbhe\Desktop\workspace_1\travelara-cd-v2


In [4]:
from __future__ import annotations
import pandas as pd
from app.schemas import (
    PlanningRequest, POI, StructuredIntent, Itinerary, DayPlan, ItineraryStop, ItineraryScore
)
from app.clustering.cluster import (
    haversine_km, cluster_pois, score_all_pois
)
from app.config import settings
from app.utils.save import save_artifact
from app.providers.provider import run_retrieval

from app.clustering.filter import *

from app.details.wikidata import *

In [5]:
from app.clustering.cluster import select_clusters
from app.details.wikidata import *

In [6]:
from app.schemas import (
    StructuredIntent,
    Preferences,
    Constraints,
)

intent = StructuredIntent(
    destination="Tokyo",
    days=5,
    stay_location="Shinjuku",
    is_international=True,
    budget="medium",
    preferences=Preferences(
        museums=0.8,
        food=0.8,
        nightlife=0.0,
        nature=0.0,
        shopping=0.0,
        arts=0.0,
        history=0.0,
        wellness=0.0,
    ),
    constraints=Constraints(
        walking_limit_km=5.5,
        must_visit=[],
        avoid=[],
        budget_per_day_usd=None,
    ),
)

In [7]:
# import json
# with open(r'C:\Users\kuchbhe\Desktop\workspace_1\travelara-cd-v2\experiments\runs\123456\POIS.json', 'r', encoding='utf-8') as f:
#     data = json.load(f)

# pois = [POI.model_validate(item) for item in data]

pois_GA, lat, lon = await run_retrieval(
    source="GA",
    intent=intent,
    debug=True
)

pois_FS, lat, lon = await run_retrieval(
    source="FS",
    intent=intent,
    debug=True
)
pois = pois_GA + pois_FS
save_artifact('123456', 'POIS', pois)
sorted_pois = score_all_pois(pois, intent)

selected_clusters, selected_pois, cluster_map = select_clusters(pois, intent)


=== RETRIEVAL START ===
Provider: GA
Destination: Tokyo
Coordinates: (35.6895, 139.69171)
Preferences: {'museums': 0.8, 'food': 0.8, 'nightlife': 0.0, 'nature': 0.0, 'shopping': 0.0, 'arts': 0.0, 'history': 0.0, 'wellness': 0.0}
Processed: museums
Processed: food

=== DEDUPLICATION ===
Input POIs: 847
Output POIs: 755
Dropped: 92
By Category: {'museums': 3, 'food': 89}

=== AFTER DEDUP ===
Total POIs: 755
{'museums': 352, 'food': 403}

=== MUST VISIT FILTER ===
Must Visit Targets: []
Matched POIs: 0

=== AVOID FILTER ===
Avoid Categories: set()
Removed: 0

=== FINAL RESULT ===
Total POIs: 755
Must Visit POIs: 0
Regular POIs: 755
{'museums': 352, 'food': 403}


=== RETRIEVAL START ===
Provider: FS
Destination: Tokyo
Coordinates: (35.6895, 139.69171)
Preferences: {'museums': 0.8, 'food': 0.8, 'nightlife': 0.0, 'nature': 0.0, 'shopping': 0.0, 'arts': 0.0, 'history': 0.0, 'wellness': 0.0}
Processed: museums
Processed: food

=== DEDUPLICATION ===
Input POIs: 98
Output POIs: 49
Dropped: 49


In [8]:
await enrich_selected_pois(selected_pois)

In [9]:
from __future__ import annotations

import torch
import torch.nn.functional as F

from sentence_transformers import SentenceTransformer

from app.schemas import POI, StructuredIntent


class SemanticScorer:
    def __init__(
        self,
        model_name: str = "BAAI/bge-m3",
        device: str | None = None,
    ):
        self.model = SentenceTransformer(
            model_name,
            device=device,
        )

    @staticmethod
    def _build_user_profile(
        intent: StructuredIntent,
    ) -> str:

        prefs = []

        for name, value in intent.preferences.model_dump().items():
            if value > 0:
                prefs.append(f"{name} ({value:.2f})")

        avoids = intent.constraints.avoid or []

        return f"""
Destination: {intent.destination}

Stay location:
{intent.stay_location}

Budget:
{intent.budget}

Trip length:
{intent.days} days

Walking limit:
{intent.constraints.walking_limit_km} km

Interested in:
{", ".join(prefs)}

Avoid:
{", ".join(avoids)}
""".strip()

    @staticmethod
    def _build_poi_document(
        poi: POI,
    ) -> str:

        wiki = ""

        if poi.wiki_enrichment:
            wiki = (
                poi.wiki_enrichment.get("description")
                or ""
            )

        tags = ", ".join(poi.tags or [])

        return f"""
Name:
{poi.name}

Category:
{poi.category}

Tags:
{tags}

Description:
{wiki}
""".strip()

    def score(
        self,
        pois: list[POI],
        intent: StructuredIntent,
    ) -> list[float]:

        if not pois:
            return []

        user_doc = self._build_user_profile(intent)

        poi_docs = [
            self._build_poi_document(p)
            for p in pois
        ]

        user_embedding = self.model.encode(
            user_doc,
            convert_to_tensor=True,
            normalize_embeddings=True,
        )

        poi_embeddings = self.model.encode(
            poi_docs,
            convert_to_tensor=True,
            normalize_embeddings=True,
            batch_size=64,
        )

        scores = (
            F.cosine_similarity(
                poi_embeddings,
                user_embedding.unsqueeze(0),
            )
            .cpu()
            .tolist()
        )

        # normalize [-1,1] -> [0,1]
        scores = [
            (s + 1) / 2
            for s in scores
        ]

        return scores


def score_semantics(
    pois: list[POI],
    intent: StructuredIntent,
) -> list[POI]:

    scorer = SemanticScorer()

    scores = scorer.score(
        pois,
        intent,
    )

    for poi, score in zip(
        pois,
        scores,
    ):
        poi.anchor_score.semantic_score=score

    return pois

c:\Users\kuchbhe\Desktop\workspace_1\leadscoring\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
selected_pois = score_semantics(
    selected_pois,
    intent,
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 85047.60it/s]


In [11]:
from collections import defaultdict
import math
import numpy as np
from app.clustering.cluster import _normalize, haversine_m

def compute_anchor_scores(
    pois: list[POI],
    cluster_map: dict[str, int],
    sigma_m: float = 500.0,
    neighbor_radius_m: float = 400.0,
) -> list[POI]:

    clusters = defaultdict(list)

    for poi in pois:
        clusters[cluster_map[poi.id]].append(poi)

    utility = _normalize({
        p.id: p.utility_score.raw_score
        for p in pois
    })

    for members in clusters.values():

        if len(members) == 1:
            poi = members[0]
            poi.anchor_score.representative_score = 1.0
            poi.anchor_score.expansion_score = 1.0
            poi.anchor_score.connectivity_score = 1.0
            poi.anchor_score.importance_score = poi.popularity_score or 0.0
            poi.anchor_score.overall_anchor = (
                0.30 * poi.anchor_score.semantic_score +
                0.25 * poi.anchor_score.representative_score +
                0.20 * poi.anchor_score.expansion_score +
                0.15 * poi.anchor_score.connectivity_score+
                0.05 * poi.anchor_score.importance_score 
            )
            continue

        centroid_lat = np.mean([p.lat for p in members])
        centroid_lon = np.mean([p.lon for p in members])

        max_centroid_dist = max(
            haversine_m(centroid_lat, centroid_lon, p.lat, p.lon)
            for p in members
        ) + 1e-6

        represent = {}
        expansion = {}
        connectivity = {}
        importance = {}

        for poi in members:

            represent[poi.id] = 1.0 - (
                haversine_m(
                    centroid_lat,
                    centroid_lon,
                    poi.lat,
                    poi.lon,
                ) / max_centroid_dist
            )

            expansion_score = 0.0
            connectivity_score = 0

            for other in members:

                if other.id == poi.id:
                    continue

                d = haversine_m(
                    poi.lat,
                    poi.lon,
                    other.lat,
                    other.lon,
                )

                expansion_score += (
                    utility[other.id] *
                    math.exp(-d / sigma_m)
                )

                if d <= neighbor_radius_m and utility[other.id] >= 0.6:
                    connectivity_score += 1

            expansion[poi.id] = expansion_score
            connectivity[poi.id] = connectivity_score
            importance[poi.id] = poi.popularity_score or 0.0

        represent = _normalize(represent)
        expansion = _normalize(expansion)
        connectivity = _normalize(connectivity)
        importance = _normalize(importance)

        for poi in members:

            poi.anchor_score.representative_score = represent[poi.id]
            poi.anchor_score.expansion_score = expansion[poi.id]
            poi.anchor_score.connectivity_score = connectivity[poi.id]
            poi.anchor_score.importance_score = importance[poi.id]

            poi.anchor_score.overall_anchor = (
                0.30 * poi.anchor_score.semantic_score +
                0.25 * utility[poi.id] +
                0.20 * represent[poi.id] +
                0.15 * expansion[poi.id] +
                0.05 * connectivity[poi.id] +
                0.05 * importance[poi.id]
            )

    return pois

In [10]:
selected_pois_new = compute_anchor_scores(selected_pois, cluster_map)

In [ ]:
def candidate_score(anchor: POI, poi: POI) -> float:
    distance = haversine_m(anchor.lat, anchor.lon, poi.lat, poi.lon)
    distance_score = math.exp(-distance / 500.0)

    return (
        0.60 * poi.utility_score.overall_score +
        0.25 * poi.anchor_score.semantic_score +
        0.15 * distance_score
    )


from collections import defaultdict
import math

def build_candidate_pool(
    pois: list[POI],
    cluster_metrics: list[dict],
    cluster_map: dict[str, int],
    days: int,
    target_per_cluster: int = 6,
    expansion_radius_m: float = 600,
):

    clusters = defaultdict(list)

    for poi in pois:
        clusters[cluster_map[poi.id]].append(poi)

    ranked_clusters = sorted(
        cluster_metrics,
        key=lambda c: c["survival_score"],
        reverse=True,
    )[:days]

    used = set()
    final_days = []

    for cluster in ranked_clusters:

        cid = cluster["cluster_id"]
        members = clusters[cid]

        if not members:
            continue

        members.sort(
            key=lambda p: p.anchor_score.overall_anchor,
            reverse=True,
        )

        anchor = members[0]
        used.add(anchor.id)

        candidates = [anchor]

        remaining = sorted(
            members[1:],
            key=lambda p: candidate_score(anchor, p),
            reverse=True,
        )

        for poi in remaining:

            if len(candidates) >= target_per_cluster:
                break

            if poi.id in used:
                continue

            candidates.append(poi)
            used.add(poi.id)

        if len(candidates) < target_per_cluster:

            nearby = [
                p for p in pois
                if p.id not in used
                and cluster_map[p.id] != cid
                and haversine_m(anchor.lat, anchor.lon, p.lat, p.lon) <= expansion_radius_m
            ]

            nearby.sort(
                key=lambda p: candidate_score(anchor, p),
                reverse=True,
            )

            for poi in nearby:

                if len(candidates) >= target_per_cluster:
                    break

                candidates.append(poi)
                used.add(poi.id)

        final_days.append({
            "cluster": cluster,
            "anchor": anchor,
            "pois": candidates,
        })

    return final_days

[POI(id='GA_51b256420f4a76614059ec6dc183a6d84140f00102f9017ba01f220000000092030e534f4d504fe7be8ee8a193e9a4a8', name='SOMPO美術館', lat=35.692581624458256, lon=139.69654047924774, category='museums', tags=['building', 'building.tourism', 'entertainment', 'entertainment.museum'], popularity_score=0.5, opening_hours='Tu-Su 10:00-18:00', external_links=['https://sompo-museum.org'], rating=3.5, reviews=None, address='Sompo Museum of Art, 1, Nishi-Shinjuku, Shinjuku, Nishi-Shinjuku 1 160-8338, Japan', pincode='160-8338', wiki_and_media={'wikidata': 'Q1614504', 'wikipedia': 'ja:SOMPO美術館', 'image': 'https://photos.app.goo.gl/vZzsgc9my7NiCAFVA'}, distance=555, source='geoapify', utility_score=QualityScore(id='GA_51b256420f4a76614059ec6dc183a6d84140f00102f9017ba01f220000000092030e534f4d504fe7be8ee8a193e9a4a8', name_score=1.0, source_score=0.0, tag_score=2.6000000000000005, external_link_score=0.5, wiki_score=0.22629438553091683, semantic_score=0.10312174542620299, overall_score=0.9547309853348845, 

name_score=1.0, source_score=0.0, tag_score=2.6000000000000005, external_link_score=0.5, wiki_score=0.22629438553091683, semantic_score=0.10312174542620302, overall_score=0.9547309853348845, raw_score=6.048806814574

Inefficiancies in the utility score, low semantic score, and yet scored high on overall score.